# 18 Audit Dataset Benchmark


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=True)


In [ ]:
from pathlib import Path
import json
import os
import sys
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next(
    (p.resolve() for p in _candidates
     if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()),
    None
)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')

LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('18_Audit_Dataset_Benchmark')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

for _name in ['experiment_lib', 'care_km', 'generalization_data']:
    sys.modules.pop(_name, None)
import experiment_lib as el
import care_km as ck
import generalization_data as gd


METHODS = list(el.METHODS)
CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print('PAPER_ROOT:', PAPER_ROOT)
print('DATA_DIR  :', DATA_DIR)
print('METHODS   :', METHODS)
print('CARE config:', CARE_CONFIG)


DATASETS = gd.resolve_dataset_keys()
EXPECTED_DEFAULT = gd.DATASET_ORDER.copy()

if os.environ.get('CARE_KM_GEN_DATASETS', '').strip() == '':
    if DATASETS != EXPECTED_DEFAULT:
        raise RuntimeError('Urutan registry dataset generalization berubah.')

rows = []
for key in DATASETS:
    spec = gd.DATASETS[key]
    source = DATA_DIR / spec['file']
    if not source.exists():
        raise FileNotFoundError(source)

    print(f'Audit: {key} <- {source.name}', flush=True)
    row = gd.audit_dataset(DATA_DIR, key)
    row['source_sha256'] = sha256_file(source)
    rows.append(row)

audit = pd.DataFrame(rows)

if audit['dataset'].duplicated().any():
    raise RuntimeError('Dataset duplikat pada audit.')
if not audit['valid'].astype(bool).all():
    bad = audit.loc[~audit['valid'].astype(bool), 'dataset'].tolist()
    raise RuntimeError(f'Dataset structurally invalid: {bad}')

audit.to_csv(artifact_path('09_generalization_dataset_audit.csv','18_Audit_Dataset_Benchmark'), index=False)

source_hashes = {
    gd.DATASETS[k]['file']: sha256_file(DATA_DIR / gd.DATASETS[k]['file'])
    for k in DATASETS
}
provenance = {
    'source_location': 'REVISED/DATA',
    'dataset_count': int(len(DATASETS)),
    'dataset_keys': DATASETS,
    'methods': METHODS,
    'resampling_repetitions': 30,
    'resampling_fraction': 0.80,
    'matched_resampling': True,
    'without_replacement': True,
    'care_config': CARE_CONFIG,
    'care_metadata': el.frozen_care_metadata(),
    'source_files_sha256': source_hashes,
    'external_generalization_folder_used': False,
}
(artifact_path('09_generalization_provenance.json','18_Audit_Dataset_Benchmark')).write_text(
    json.dumps(provenance, indent=2), encoding='utf-8'
)

display(audit)
warnings = audit.loc[~audit['metadata_match'].astype(bool), [
    'dataset','rows','features','classes',
    'reference_rows','reference_features','reference_k'
]]
if len(warnings):
    print('\nMetadata reference warning(s); structural audit tetap valid:')
    display(warnings)

print(json.dumps(provenance, indent=2))
print('STATUS: PASS')
